In [1]:
with open('kegg_id.txt', 'r') as f:
    f = f.read()
    kegg_ids = f.split('\n')

print(kegg_ids)

['hsa:3107', 'hsa:3105', 'hsa:3106', 'hsa:3135', 'hsa:3133']


In [2]:
import requests

In [10]:
def getDataFromKegg(operation, argument):
    url = f"https://rest.kegg.jp/{operation}/{argument}"

    resp = requests.get(
        url
    )

    if resp.ok:
        return resp.text
    
def retrievePositionFromKegg(kegg_id):
    data = getDataFromKegg('get', kegg_id)

    position = data.split('POSITION')[1].split('MOTIF')[0].split(':')

    chromosom = position[0].strip()
    pozycja = position[1].strip('\n')

    return chromosom, pozycja
    

for kegg_id in kegg_ids:
    chromosom, pozycja = retrievePositionFromKegg(kegg_id)
    print(f'{kegg_id}, chromosom {chromosom}, pozycja {pozycja}')

hsa:3107, chromosom 6, pozycja complement(31268749..31272092)
hsa:3105, chromosom 6, pozycja 29942532..29945870
hsa:3106, chromosom 6, pozycja complement(31353875..31357179)
hsa:3135, chromosom 6, pozycja 29826474..29831021
hsa:3133, chromosom 6, pozycja 30489509..30494194


In [12]:
with open('genes_fasta', 'w') as f:
    for i in range(3):    
        f.write(getDataFromKegg('get', f'{kegg_ids[i]}/ntseq'))

In [14]:
import time

def runClustal(sequecnes_in_fasta):
    resp = requests.post(
        'https://www.ebi.ac.uk/Tools/services/rest/clustalo/run',
        data={
            "email": "marcel.thiel@ug.edu.pl",
            "sequence": sequecnes_in_fasta,
            "outfmt": "fa"
        }
    )
    jobid = resp.text

    return jobid


def getStatus(jobId):
    url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/status/{jobId}"
    
    resp = requests.get(
        url
    )

    return resp.text


def getResults(jobId):
    url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/result/{jobId}/fa"

    resp = requests.get(
        url
    )

    return resp.text

with open('genes_fasta', 'r') as f:
    f = f.read()

    jobid = runClustal(f)
    
    for i in range(20):
        status = getStatus(jobid)
        if status == "FINISHED":
            msaResults = getResults(jobid)
            break
        else:
            time.sleep(1)

    print(msaResults)

>3107 K06751 MHC class I antigen | (RefSeq) HLA-C, D6S204, HLA-JY3, HLAC, HLC-C, MHC, PSORS1; major histocompatibility complex, class
atgcgggtcatggcgccccgagccctcctcctgctgctctcgggaggcctggccctgacc
gagacctgggcctgctcccactccatgaggtatttcgacaccgccgtgtcccggcccggc
cgcggagagccccgcttcatctcagtgggctacgtggacgacacgcagttcgtgcggttc
gacagcgacgccgcgagtccgagaggggagccgcgggcgccgtgggtggagcaggagggg
ccggagtattgggaccgggagacacagaagtacaagcgccaggcacaggctgaccgagtg
agcctgcggaacctgcgcggctactacaaccagagcgaggacgggtctcacaccctccag
aggatgtctggctgcgacctggggcccgacgggcgcctcctccgcgggtatgaccagtcc
gcctacgacggcaaggattacatcgccctgaacgaggacctgcgctcctggaccgccgcg
gacaccgcggctcagatcacccagcgcaagttggaggcggcccgtgcggcggagcagctg
agagcctacctggagggcacgtgcgtggagtggctccgcagatacctggagaacgggaag
gagacgctgcagcgcgcagaacccccaaagacacacgtgacccaccaccccctctctgac
catgaggccaccctgaggtgctgggccctgggcttctaccctgcggagatcacactgacc
tggcagcgggatggggaggaccagacccaggacaccgagcttgtggagaccaggccagca
ggagatggaaccttccagaagtgggcagctgtggtggtgccttctggacaagagcagaga
tacacgtgccat

In [19]:
def getPdbInfo(pdb_id):
    
    url = f"https://files.rcsb.org/view/{pdb_id}.pdb"

    resp = requests.get(url)

    if resp.ok:
        data = resp.text

    return data


def pktC(pdb_id):
    data = getPdbInfo(pdb_id)

    header = data.split('TITLE')[0].strip('\n').strip()

    helixCount = 0
    sheetCount = 0
    atomCount = 0

    for line in data.split('\n'):
        if line.startswith('HELIX'):
            helixCount += 1
        elif line.startswith('SHEET'):
            sheetCount += 1
        elif line.startswith('ATOM'):
            atomCount += 1

    return header, helixCount, sheetCount, atomCount

#print(getPdbInfo('1LKX'))
print(pktC('1LKX'))

('HEADER    CONTRACTILE PROTEIN                     26-APR-02   1LKX', 127, 58, 21069)
